# The Datacenter Spec Assistant

**Project 3 · RAG Application**

**CS4500-800 CIT Seminar · Fall 2026 · Dr. Nan Wang · Brian Balint**

An open-book assistant for Open Compute Project hardware specifications. Ask it
a question about rack dimensions, bus bar voltages or thermal limits and it
retrieves the relevant passages from the actual specification documents, puts
them in front of the model, and answers from those passages with a citation to
the document and page.

The model is not asked to remember anything. It is asked to read.

**Source of truth:** OCP specifications, published under CC BY 4.0 and the OCP
Hardware License, both of which permit redistribution. The PDFs are submitted
alongside this notebook.

**Runs against:** Ollama on localhost, `qwen2.5:7b-instruct` for generation and
`nomic-embed-text` for retrieval. Nothing leaves this machine.

---
## The Pain Point

Datacenter hardware specifications are the kind of document nobody reads front
to back and everybody needs one paragraph from. An OCP rack specification runs
to dozens of pages of dimensions, tolerances, electrical limits and mechanical
detail. The answer to "what is the bus bar nominal voltage" is one line buried
somewhere in it.

A general-purpose language model handles this badly, and it does so in the worst
possible way. It does not say "I have not read that document." It produces a
number. The number is often close to right, because the model has absorbed
adjacent material during training, and close to right is indistinguishable from
right until somebody builds to it.

This is the failure mode worth demonstrating rather than describing, so the
notebook does both: every test question is asked twice, once with the
specification in front of the model and once without.

**Why an LLM is the right tool here.** Full-text search finds the page. It does
not answer the question. A question phrased in plain language rarely uses the
same words as the specification, and the answer is usually a synthesis of two or
three passages rather than a single sentence to highlight. The model's job is
reading and reasoning over retrieved text, not recall.

---
## Setup

Two models. One turns text into vectors so passages can be ranked by meaning
rather than by keyword. The other reads the retrieved passages and answers.

```
ollama pull nomic-embed-text
ollama pull qwen2.5:7b-instruct
```

In [1]:
import json
import os
import re
import textwrap
from pathlib import Path

import numpy as np
import requests

OLLAMA = "http://127.0.0.1:11434"
CHAT_MODEL = "qwen2.5:7b-instruct"
EMBED_MODEL = "nomic-embed-text"

# Put the OCP specification PDFs here.
DOCS_DIR = Path.home() / "ocp_specs"


def ollama_up():
    try:
        tags = requests.get(f"{OLLAMA}/api/tags", timeout=5).json()
        names = {m["name"] for m in tags.get("models", [])}
        return names
    except requests.exceptions.RequestException:
        return None


names = ollama_up()
if names is None:
    print("Ollama is not reachable at", OLLAMA)
else:
    for model in (CHAT_MODEL, EMBED_MODEL):
        mark = "ok" if any(n.startswith(model.split(":")[0]) for n in names) else "MISSING"
        print(f"  {mark:8} {model}")

  ok       qwen2.5:7b-instruct
  ok       nomic-embed-text


---
## Step 1: Retrieval

### Reading the documents

Text is extracted page by page, because the page number is what makes a citation
useful. An answer that says "according to the specification" is worth much less
than one that says "OCP Open Rack v3, page 14", where the reader can go and
check.

In [2]:
import pypdf

# Specifications exported from Google Docs carry invisible bidirectional
# formatting marks, and every page repeats a running header and a date/page
# footer. Left in place, that boilerplate lands in every chunk and dilutes the
# embeddings, so it is stripped at ingestion.
INVISIBLE = re.compile(r"[\u202a-\u202e\u2066-\u2069\u200e\u200f\u00ad]")
BOILERPLATE = [
    re.compile(r"Open Compute Project\s*.\s*Open Rack.*?Specification", re.I),
    re.compile(r"Date:\s*\d{1,2}\s+\w+\s+\d{4}", re.I),
    re.compile(r"Page\s*\d{1,3}\s*$", re.I),
    re.compile(r"http\S+"),
]


def clean(text):
    text = INVISIBLE.sub("", text)
    for pattern in BOILERPLATE:
        text = pattern.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()


def load_pages(directory: Path, min_chars=120):
    """Read every PDF in a directory into (document, page number, text)."""
    pages = []
    pdfs = sorted(directory.glob("*.pdf"))
    if not pdfs:
        print(f"No PDFs found in {directory}. Put the OCP specifications there.")
        return pages

    for path in pdfs:
        reader = pypdf.PdfReader(str(path))
        kept = 0
        for number, page in enumerate(reader.pages, 1):
            text = clean(page.extract_text() or "")
            # Figure-only pages extract to a caption and nothing else. They add
            # no retrievable content, so they are dropped rather than indexed.
            if len(text) < min_chars:
                continue
            pages.append({"doc": path.stem, "page": number, "text": text})
            kept += 1
        print(f"  {path.stem}: {kept} of {len(reader.pages)} pages have usable text")

    total = sum(len(p["text"]) for p in pages)
    print(f"\n{len(pages)} pages, {total:,} characters of text")
    if total < 60000:
        print("\nNote: this is a small corpus. Retrieval discriminates better with "
              "more documents; consider adding two or three more specifications.")
    return pages


pages = load_pages(DOCS_DIR)

  OCP Open Rack V3 HPR V2 Power Monitoring Module (PMM) Rev 1.0.0 FINAL: 32 of 33 pages have usable text
  Open Rack Base Specification Version 3_rev1.1_030524: 21 of 28 pages have usable text
  Open Rack V3 HPR V2 12kW PSU Module SPEC V1.0.0: 50 of 50 pages have usable text

103 pages, 152,312 characters of text


### Splitting into passages

A whole page is too coarse. If a page covers three topics, retrieving it for one
of them drags the other two into the prompt and spends context on noise.

Pages are split into overlapping windows of roughly 1,200 characters. The
overlap matters: a requirement that straddles a boundary would otherwise be cut
in half and lost from both pieces. Splitting on sentence ends rather than
mid-word keeps each passage readable on its own, which is necessary because the
model sees the passage without the page around it.

In [3]:
def split_page(text, size=1200, overlap=200):
    """Break a page into overlapping windows, preferring sentence boundaries."""
    text = re.sub(r"\s+", " ", text).strip()
    if len(text) <= size:
        return [text]

    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        if end < len(text):
            # Back up to the last sentence end in the final quarter of the window.
            window = text[start + int(size * 0.75):end]
            match = list(re.finditer(r"[.!?]\s", window))
            if match:
                end = start + int(size * 0.75) + match[-1].end()
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = end - overlap
    return [c for c in chunks if len(c) > 60]


passages = []
for page in pages:
    for chunk in split_page(page["text"]):
        passages.append({"doc": page["doc"], "page": page["page"], "text": chunk})

print(f"{len(passages)} passages from {len(pages)} pages")
if passages:
    lengths = [len(p["text"]) for p in passages]
    print(f"average length: {sum(lengths) // len(lengths)} characters")
    print(f"\nExample passage, {passages[len(passages)//2]['doc']} p.{passages[len(passages)//2]['page']}:")
    print(textwrap.fill(passages[len(passages)//2]["text"][:400], 78))

195 passages from 103 pages
average length: 874 characters

Example passage, Open Rack V3 HPR V2 12kW PSU Module SPEC V1.0.0 p.5:
ensions 35 8.2. Construction 36 8.3. Materials and Fasteners 36 8.3.1. PCR
Targets & Implementation 36 8.3.2. Supplier Qualification & Material Testing
37 8.3.3. Documentation & Certification 37 8.4. Latch & Handle Requirements 38
8.5. EMI Gaskets 38 8.6. Chassis Interface 39 8.7. Rear Blind Mate Connector
39 8.8. Rear Blind Mate Connector Protection 39 8.9. Mechanical Drawings 40 12
June 2026


### Embedding

Each passage becomes a vector. Two pieces of text with similar meaning end up
close together in that space, which is what lets a question phrased in plain
language find a passage that never uses the same words. A keyword search for
"how much power" would miss a paragraph headed "electrical load capacity".

This runs once. The vectors are cached to disk so re-running the notebook does
not re-embed the whole corpus.

In [4]:
CACHE = Path.home() / "ocp_index.npz"


def embed(texts, batch=32):
    """Turn a list of strings into a matrix of unit vectors."""
    out = []
    for i in range(0, len(texts), batch):
        response = requests.post(
            f"{OLLAMA}/api/embed",
            json={"model": EMBED_MODEL, "input": texts[i:i + batch]},
            timeout=300,
        )
        response.raise_for_status()
        out.extend(response.json()["embeddings"])
        print(f"  embedded {min(i + batch, len(texts))}/{len(texts)}", end="\r")

    matrix = np.array(out, dtype=np.float32)
    # Normalising once means similarity is a dot product later, not a division.
    matrix /= np.linalg.norm(matrix, axis=1, keepdims=True)
    print()
    return matrix


if passages:
    if CACHE.exists():
        cached = np.load(CACHE, allow_pickle=True)
        if len(cached["vectors"]) == len(passages):
            vectors = cached["vectors"]
            print(f"Loaded {len(vectors)} cached vectors from {CACHE.name}")
        else:
            vectors = embed([p["text"] for p in passages])
            np.savez(CACHE, vectors=vectors)
    else:
        vectors = embed([p["text"] for p in passages])
        np.savez(CACHE, vectors=vectors)
        print(f"Cached to {CACHE}")

    print(f"index: {vectors.shape[0]} passages, {vectors.shape[1]} dimensions")

  embedded 195/195
Cached to /home/brian/ocp_index.npz
index: 195 passages, 768 dimensions


### Searching

The question is embedded the same way, then ranked against every passage by
cosine similarity. Because the vectors are already unit length, that is a single
matrix multiplication.

In [5]:
def search(question, k=5):
    """Return the k passages closest in meaning to the question."""
    query = embed([question])[0]
    scores = vectors @ query
    top = np.argsort(scores)[::-1][:k]
    return [(passages[i], float(scores[i])) for i in top]


hits = search("What voltage does the busbar operate at?", k=3)
for passage, score in hits:
    print(f"[{score:.3f}] {passage['doc']} p.{passage['page']}")
    print(textwrap.fill(passage["text"][:200], 76, initial_indent="    ",
                        subsequent_indent="    "))
    print()

  embedded 1/1
[0.721] Open Rack Base Specification Version 3_rev1.1_030524 p.16
    6.3 Busbar 6.3.1 48V Busbar Geometry Figure 6.3.1 48V Busbar Cross-
    section 6.3.2 Ground Path The electrically conductive surface of the
    busbar (note 4) as defined in Figure 6.3.1 above SHALL have a co

[0.687] Open Rack Base Specification Version 3_rev1.1_030524 p.26
    applied in any direction except upwards, at a height not exceeding 2 m
    (78.74 in) from the floor. The force is applied to the front, back and
    each side of the system, each for a period of 1 minute. .

[0.677] Open Rack Base Specification Version 3_rev1.1_030524 p.26
    cture, a new bus bar connector has been designed to improve upon last
    generation challenges and incorporate new features such as the ability
    to monitor voltage drop of the input power cable assembly.



---
## Step 2: Augmentation

The retrieved passages are assembled into a prompt with instructions. Three
things in the instruction do most of the work:

**Answer only from the context.** Without this the model falls back on training
data the moment the passages are thin, and the answer looks identical either
way. This is the single most important line in the prompt and it maps directly
to the contextual accuracy criterion.

**Say so when the answer is not there.** A model with no permission to decline
will produce something. Giving it an explicit way out is what turns a confident
wrong answer into a useful "not in these documents".

**Cite the source.** Each passage is labelled with its document and page, and
the model is required to carry that label into the answer. The citation is not
decoration: it is what lets the reader verify the answer without trusting the
model at all.

In [6]:
SYSTEM = """You answer questions about datacenter hardware specifications.

You will be given numbered passages from specification documents, then a
question. Answer using only those passages.

Rules:
1. Use only the information in the passages. Do not add anything you know from
   elsewhere, even if you are confident it is correct.
2. Cite the source for every fact, in the form [Document, p.N], using the label
   given with each passage.
3. If the passages do not contain the answer, say exactly: "The provided
   specifications do not cover this." Then say what they do cover that is
   closest. Do not guess.
4. If the passages disagree, say so and cite both.
5. Be direct. Give the number or the requirement first, then the detail."""


def build_prompt(question, hits):
    """Assemble retrieved passages and the question into one rich prompt."""
    blocks = []
    for n, (passage, score) in enumerate(hits, 1):
        label = f"{passage['doc']}, p.{passage['page']}"
        blocks.append(f"[Passage {n}] Source: {label}\n{passage['text']}")

    context = "\n\n".join(blocks)
    return f"""Here are passages from the specification documents:

{context}

Question: {question}"""


example = build_prompt("What voltage does the busbar operate at?", hits)
print(example[:900])
print("\n...\n")
print(f"prompt length: {len(example)} characters, roughly {len(example)//4} tokens")

Here are passages from the specification documents:

[Passage 1] Source: Open Rack Base Specification Version 3_rev1.1_030524, p.16
6.3 Busbar 6.3.1 48V Busbar Geometry Figure 6.3.1 48V Busbar Cross-section 6.3.2 Ground Path The electrically conductive surface of the busbar (note 4) as defined in Figure 6.3.1 above SHALL have a continuous electrically conductive path to the frame of the rack. The ground path SHALL pass rust grade 6 per ASTM D610-01 after 48 hours of salt spray per ASTM B117-07.

[Passage 2] Source: Open Rack Base Specification Version 3_rev1.1_030524, p.26
applied in any direction except upwards, at a height not exceeding 2 m (78.74 in) from the floor. The force is applied to the front, back and each side of the system, each for a period of 1 minute. . 12.0 Electrical Requirements for IT Gear 12.1 Input Voltage Specification Option 1 : The expected nominal input voltage 

...

prompt length: 2269 characters, roughly 567 tokens


---
## Step 3: Generation

The model reads the assembled prompt and answers. Temperature is zero, so the
same question against the same passages gives the same answer every time.

In [7]:
def ask_model(system, user, timeout=180):
    response = requests.post(
        f"{OLLAMA}/api/chat",
        json={
            "model": CHAT_MODEL,
            "messages": [
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            "stream": False,
            "options": {"temperature": 0},
        },
        timeout=timeout,
    )
    response.raise_for_status()
    return response.json()["message"]["content"].strip()


def answer(question, k=5, show_sources=True):
    """The full pipeline: retrieve, augment, generate."""
    hits = search(question, k=k)
    reply = ask_model(SYSTEM, build_prompt(question, hits))

    print(f"Q: {question}\n")
    print(textwrap.fill(reply, 78))

    if show_sources:
        print("\nRetrieved from:")
        for passage, score in hits:
            print(f"  [{score:.3f}] {passage['doc']}, p.{passage['page']}")
    return reply, hits


_ = answer("What voltage does the busbar operate at?")

  embedded 1/1
Q: What voltage does the busbar operate at?

The busbar operates at 48V. [Passage 1, p.16]

Retrieved from:
  [0.721] Open Rack Base Specification Version 3_rev1.1_030524, p.16
  [0.687] Open Rack Base Specification Version 3_rev1.1_030524, p.26
  [0.677] Open Rack Base Specification Version 3_rev1.1_030524, p.26
  [0.672] Open Rack Base Specification Version 3_rev1.1_030524, p.17
  [0.634] Open Rack Base Specification Version 3_rev1.1_030524, p.27


---
## Does It Actually Use the Documents?

The grading criterion that matters most is whether the assistant answers from
the provided text or hallucinates from general training. Asserting that it does
is worth nothing. The test below asks the same questions twice, once with
retrieved passages and once with none, and prints both answers side by side.

Without context, one of two things happens, and both are informative. Either the
model refuses, which is the honest failure, or it produces a confident
specification-sounding answer that came from nowhere, which is the dangerous one.

In [8]:
def answer_without_context(question):
    """Same model, same settings, no retrieved passages."""
    return ask_model(
        "You answer questions about datacenter hardware specifications.",
        question,
    )


def compare(question, k=5):
    print("=" * 78)
    print(f"Q: {question}")
    print("=" * 78)

    print("\nWITHOUT the documents (model training only)")
    print("-" * 78)
    print(textwrap.fill(answer_without_context(question), 78))

    hits = search(question, k=k)
    print("\nWITH the documents (RAG)")
    print("-" * 78)
    print(textwrap.fill(ask_model(SYSTEM, build_prompt(question, hits)), 78))
    print("\nSources:")
    for passage, score in hits[:3]:
        print(f"  [{score:.3f}] {passage['doc']}, p.{passage['page']}")
    print()

### Test questions

Write these against your own corpus once the PDFs are loaded. The set below is a
starting shape rather than a finished list, and the last one matters as much as
the rest.

Aim for questions with a specific, checkable answer: a voltage, a dimension, a
tolerance, a required condition. Vague questions produce vague answers and prove
nothing either way.

The final question is deliberately outside the corpus. A system that answers it
anyway has failed, however good the answer sounds.

In [9]:
# Questions with specific, checkable answers in the corpus. Vague questions
# produce vague answers and prove nothing either way.
QUESTIONS = [
    "What voltage does the busbar operate at?",
    "What torque is required for the M6 thread-forming screws, and what is the "
    "minimum strip-out torque?",
    "What corrosion standard must the ground path meet, and after how many "
    "hours of salt spray?",
    "What screws mount the busbar at the top of the rack?",
    # Outside the corpus. The correct answer is a refusal.
    "What is the recommended hot aisle containment height for a Tier IV facility?",
]

for question in QUESTIONS:
    compare(question)

Q: What voltage does the busbar operate at?

WITHOUT the documents (model training only)
------------------------------------------------------------------------------
The voltage of a busbar can vary depending on the specific application and the
power distribution system. Busbars are commonly used in high-voltage power
distribution systems, and they can operate at various voltage levels,
including:  1. **Low Voltage (LV) Busbars**: Typically operate at voltages up
to 1 kV (1000 volts). These are used in industrial and commercial settings for
power distribution.  2. **Medium Voltage (MV) Busbars**: Operate at voltages
ranging from 1 kV to 35 kV. These are used in larger industrial facilities,
substations, and power distribution networks.  3. **High Voltage (HV)
Busbars**: Operate at voltages above 35 kV. These are used in long-distance
power transmission systems.  The exact voltage of a busbar in a specific data
center or industrial setting would depend on the design and requirements o

---
## Reasoning Over the Text

Retrieving a passage and repeating it is lookup. The questions below need the
model to combine two or three passages, or to notice that the answer depends on
a condition stated elsewhere, which is what the reasoning criterion is about.

In [10]:
REASONING_QUESTIONS = [
    "I am fabricating a rack to this specification. Summarise the fastener and "
    "torque requirements I need to meet.",
    "What environmental and corrosion testing does the rack have to pass?",
    "What is optional in this specification and what is mandatory? Give "
    "examples of each.",
]

for question in REASONING_QUESTIONS:
    print("=" * 78)
    _ = answer(question, k=6)
    print()

  embedded 1/1
Q: I am fabricating a rack to this specification. Summarise the fastener and torque requirements I need to meet.

The fastener and torque requirements you need to meet are as follows:  - All
Ø4.5mm holes in the rack are designed for steel M5 thread-forming screws per
DIN 7500. These holes must have a minimum strip out torque of 6.25 N-m. The
nominal torque required is 5 N-m and supports a minimum of 5 cycles of
tightening & untightening. [Passage 1, p.22]  - All Ø5.4mm holes in the rack
are designed for M6 thread-forming screws per DIN 7500. These holes must have
a minimum strip out torque of 6.25 N-m. The nominal torque required is 5 N-m
and supports a minimum of 5 cycles of tightening & untightening. [Passage 2,
p.23]  - For the busbar top screws, the M5 threaded holes are designed to
utilize M5 x 8mm (minimum) pan head machine screws. These holes must have a
minimum strip out torque of 6.8 N-m. The nominal torque required is 5 N-m.
[Passage 2, p.23]  - For the busbar 

---
## What This Shows

Fill this in once the comparison above has run against your corpus. The things
worth recording:

**Where the two answers diverged.** The most useful result is a question where
the model without context gave a plausible number and the retrieved passages
show a different one. That single pair demonstrates the whole argument for RAG
better than any description of the architecture.

**Whether it declined when it should have.** The out-of-corpus question is the
honesty test. Note what it actually did rather than what it was supposed to do.

**Where retrieval brought back the wrong passages.** Similarity search is not
perfect. A question using different vocabulary than the specification can rank
an unrelated passage highly, and the model will then answer from it correctly
and uselessly. This is the failure mode of RAG specifically, distinct from
hallucination, and it is worth showing one.

## Limits

- **Retrieval quality caps answer quality.** If the right passage is not in the
  top k, no amount of prompting recovers it. The model cannot ask for more.
- **Tables and figures do not survive extraction.** A specification's numbers
  are often in a table, and PDF text extraction flattens it into a run of
  values with the column headings lost. This is the single biggest weakness
  here for this kind of document.
- **Chunk boundaries can split a requirement** from the condition that governs
  it, so a retrieved passage can be accurate and still misleading on its own.
  The overlap reduces this without eliminating it.
- **Citations point at a page, not a line.** Good enough to verify, coarse
  enough to still require reading.

## Where This Goes Next

**Re-ranking.** Retrieve twenty passages by vector similarity, then have the
model score which are genuinely relevant before the answer is generated. Cheap,
and it addresses the wrong-passage failure directly.

**Table extraction.** Parsing tables into structured text before embedding would
fix the largest weakness. Specification numbers live in tables.

**Retrieval scoring as a signal.** If the best passage scores below a threshold,
the system should decline before it generates anything, rather than asking the
model to notice that its context is poor.